# Reasoning Techniques

Reasoning techniques give a model more structure when a problem cannot be answered reliably in one step.


## What to learn

- Decomposition: split a large problem into smaller questions.
- Self-consistency: try more than one solution and compare the answers.
- Search: explore alternatives when an early choice may lead down the wrong path.
- Verification: check the proposed answer with tests, tools, or explicit rules.

## Decision rule

Start with the simplest direct prompt. Add decomposition or extra attempts only when evaluation shows that they improve accuracy enough to justify the added cost.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Self-consistency and a toy tree-of-thought search, both runnable.

The fake sampler simulates the answer distribution of a mid-tier model on
a multi-step problem: mostly right, sometimes wrong in varied ways.
"""
# Import the dependencies used by this example.
import random
from collections import Counter

# --- 1. self-consistency: sample k paths, majority vote -------------------
# Define the data shape and small operations before running them.
def sample_answer(rng):
    # 60% correct answer, 40% spread across distinct wrong answers
    return rng.choices(["42", "38", "56", "40"], weights=[60, 15, 15, 10])[0]


def self_consistency(k, seed=7):
    rng = random.Random(seed)
    answers = [sample_answer(rng) for _ in range(k)]
    votes = Counter(answers)
    best, n = votes.most_common(1)[0]
    return {"answer": best, "agreement": n / k, "votes": dict(votes)}


for k in (1, 5, 15):
    r = self_consistency(k)
    print(f"k={k:>2}: answer={r['answer']} agreement={r['agreement']:.0%} votes={r['votes']}")
# Single sample is wrong 40% of the time; the k=15 vote is reliably right,
# and 'agreement' doubles as a confidence signal.

# --- 2. tree-of-thought: propose / evaluate / prune / backtrack ------------
# Toy task: build the sequence [2, 4, 8, 16] by proposing next numbers.
TARGET = [2, 4, 8, 16]


def propose(state):                      # generator: candidate next steps
    last = state[-1] if state else 1
    return [state + [last * 2], state + [last + 2], state + [last * 3]]


def evaluate(state):                     # evaluator: sure / maybe / dead-end
    if state == TARGET[: len(state)]:
        return "sure"
    return "dead-end"


def tot_search(beam_width=2, max_depth=4):
    frontier, explored = [[2]], 0
    for depth in range(max_depth - 1):
        candidates = [s for st in frontier for s in propose(st)]
        explored += len(candidates)
        scored = [(s, evaluate(s)) for s in candidates]
        frontier = [s for s, v in scored if v == "sure"][:beam_width]  # prune
        if not frontier:
            return None, explored
    return frontier[0], explored


solution, explored = tot_search()
print(f"\nToT found {solution} after evaluating {explored} candidates "
      "(pruned dead ends instead of enumerating all 27 paths)")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
